In [ ]:
import os
import warnings
import logging

# ===== SILENCE ALL WARNINGS =====
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

# Suppress Python warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Suppress TensorFlow logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

# Now import
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/lstm-method/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

# TensorFlow imports with suppressed logging
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
import importlib

# Final suppression of retracing warnings (these are annoying but harmless)
tf.keras.backend.clear_session()

In [ ]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

In [ ]:
model_used = 'gru'

model_run_name = f'{model_used}_v2'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
dc_offset_max = 2
pipe_name = 'imu_extractor'

linear_edges = np.arange(0, 51, 10)
log_edges    = np.logspace(np.log10(0.5), np.log10(50), num=10)
custom_edges = np.array([0, 1, 2, 4, 8, 15, 25, 50])

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

tof_1 = [f'tof_1_v{j}' for j in range(64)]
tof_2 = [f'tof_2_v{j}' for j in range(64)]
tof_3 = [f'tof_3_v{j}' for j in range(64)]
tof_4 = [f'tof_4_v{j}' for j in range(64)]
tof_5 = [f'tof_5_v{j}' for j in range(64)]

some_sequences = ['SEQ_000007', 'SEQ_000008', 'SEQ_000013', 'SEQ_000034', 'SEQ_000046', 'SEQ_000053']

orientation_cols = [
    'Seated Straight',
    'Lie on Side - Non Dominant',
    'Seated Lean Non Dom - FACE DOWN',
    'Lie on Back'
]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report   = False
save_model  = False
random_search = False

pipe_name = 'temporal_extractor'

In [ ]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.2,
    test_pct=0.2
)

# some_sequences = train_sample_df['sequence_id'].unique()[:50]
# train_sample_df = train_sample_df[train_sample_df['sequence_id'].isin(some_sequences)]

In [ ]:
importlib.reload(utils)
num_pattern  = 'acc|rot|thm|tof'

# In your notebook, after defining columns
raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns
)

preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), make_column_selector(pattern="acc|rot|thm|tof")),
], remainder="drop", verbose_feature_names_out=False)
preprocessor.set_output(transform="pandas")

sequence_builder = utils.SequencePadder(maxlen=60, padding_value=-999.0)  # ← your new window size

temporal_clf = utils.KerasRNNClassifier(
    rnn_type=model_used,
)

classifier = utils.ManyToOneWrapperRNN(estimator=temporal_clf, target="gesture_action")

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ("preprocessor", preprocessor),
    ("sequence_builder", sequence_builder),
    ("classifier", classifier),
])

cv = GroupKFold(n_splits=3)

if random_search:
    param_grid = {
        f'{pipe_name}__acc_mode': ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],  # Best generalization
        f'{pipe_name}__thm_mode': ['delta'],
        f'{pipe_name}__tof_mode': ['baseline'],
        # Sequence length - capture different gesture durations
        'sequence_builder__maxlen': [10, 20, 35, 50, 60, 80, 120],
        
        # RNN architecture size
        'classifier__estimator__rnn_units': [(48,), (64,), (96,), (128,), (32, 32), (64, 64), (128, 64, 64)],
        'classifier__estimator__dense_units': [(16,), (32,), (64,), (128,), (16, 8), (32, 16)],
        
        # Regularization to match model size
        'classifier__estimator__dropout': [0.1, 0.3],
        'classifier__estimator__learning_rate': [0.001, 0.0001],
        
        # Keep these fixed
        'classifier__estimator__bidirectional': [True],
        'classifier__estimator__class_weight_mode': ['balanced'],
        'classifier__estimator__epochs': [100],
        'classifier__estimator__patience': [10],
        'classifier__estimator__batch_size': [16, 32],
    }

else:
    param_grid = {
        # RawSequenceExtractor params
        f'{pipe_name}__acc_cols': [acc_columns],
        f'{pipe_name}__rot_cols': [rot_columns],
        f'{pipe_name}__thm_cols': [thm_columns],
        f'{pipe_name}__tof_cols': [tof_columns],
        f'{pipe_name}__acc_mode': ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode': ['delta'],
        f'{pipe_name}__tof_mode': ['baseline'],

        # SequencePadder params
        'sequence_builder__maxlen': [120],
        'sequence_builder__padding_value': [-999.0],

        # RNN params (nested under classifier__estimator__)
        'classifier__estimator__rnn_type': ['rnn'],
        'classifier__estimator__rnn_units': [(128,), (96, 32)],
        'classifier__estimator__dense_units': [(32, 16)],
        'classifier__estimator__dropout': [0.1],
        'classifier__estimator__learning_rate': [1e-4],
        'classifier__estimator__batch_size': [16],
        'classifier__estimator__epochs': [120],
        'classifier__estimator__patience': [10],
        "classifier__estimator__bidirectional": [True],
        "classifier__estimator__class_weight_mode": ["balanced"],
    }

In [ ]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=5,  # LSTM is slow, keep iterations low
                cv=cv,
                random_state=42,
                n_jobs=1,              # safer with tensorflow/keras on Kaggle
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=1,              # safer with tensorflow/keras on Kaggle
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']
        
        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)
    
    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    master_cv_results_df['model_type'] = model_used
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

In [ ]:
random_search

In [ ]:
master_cv_results_df.T